In [1]:
import os
import sys
import time
sys.path.append("..")

import mlflow
import asyncio
import pandas as pd
from dotenv import load_dotenv
from app.mosmap_api import get_mosmap_data
from app.yandex_api import get_yandex_data

assert load_dotenv("../.env")

URL = os.getenv("MOSMAP_URL")
URL_GEOCODER = os.getenv("MOSMAP_URL_GEOCODER")
API_KEY = os.getenv("MOSMAP_API_KEY")

C:\Users\Oleg\miniconda3\envs\sirius\lib\site-packages\mlflow\utils\requirements_utils.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
lat, lon = 55.757196, 37.658177

In [3]:
s = time.time()
_ = await get_mosmap_data(lat, lon, 300)
_ = await get_mosmap_data(lat, lon, 600)
_ = await get_yandex_data(lat, lon, 300, [["Кафе"], ["Ресторан", "Бар, паб"]])
_ = await get_yandex_data(lat, lon, 600, [["Кафе"], ["Ресторан", "Бар, паб"]])
print(f"{time.time() - s}s")

26.847734451293945s


In [4]:
s = time.time()
mosmap_data_1, mosmap_data_2, yandex_data_1, yandex_data_2 = await asyncio.gather(
    get_mosmap_data(lat, lon, 300),
    get_mosmap_data(lat, lon, 600),
    get_yandex_data(lat, lon, 300, [["Кафе"], ["Ресторан", "Бар, паб"]]),
    get_yandex_data(lat, lon, 600, [["Кафе"], ["Ресторан", "Бар, паб"]]),
)
print(f"{time.time() - s}s")

6.281087875366211s


In [5]:
mosmap_data_2 = mosmap_data_2.drop(['district_price', 'district_name'], axis=1)
mosmap_data = pd.concat([mosmap_data_1, mosmap_data_2], axis=1)
yandex_data = pd.concat([yandex_data_1, yandex_data_2], axis=1)

In [8]:
mosmap_data.T

,0
district_name,Басманный
n_buildings_300m,79
n_living_buildings_300m,22
n_flats_300m,1475
min_bc_distance_300m,63
...,...
"Пиццерии, суши, столовые_600",34
"Пекарни, кофейни_600",53
"Кафе, бары, рестораны_600",93
Социальные_600,5


In [9]:
yandex_data.T

,0
Кафе_300,12
"Ресторан_Бар, паб_300",13
Кафе_600,40
"Ресторан_Бар, паб_600",40
